# INGESTA — Datos limpios → MySQL (modelo estrella)

**Insumo:** `mvnd_limpio.csv` (notebook 02 ETL)  
**Destino:** base `mvnd_dw` en MySQL  
**Guía detallada:** `INGESTA_MYSQL.md`

| Paso | Archivo |
|------|----------|
| Conexión | `config/.env` + `db/conexion.py` |
| Transformación | `cargar_modelo_estrella.py` |
| **Ingesta a BD** | **`db/ingesta.py`** + **`ingesta_mysql.py`** |
| DDL tablas | `sql/01_ddl_modelo_estrella.sql` |

## Paso 0 — Dependencias

```bash
pip install -r requirements-db.txt
```

Copia `config/env.example` → `config/.env` y escribe tu contraseña MySQL.

In [1]:
%pip install pandas sqlalchemy pymysql -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Paso 1 — Probar conexión a MySQL

Aquí se lee `config/.env` y se valida que el servidor responda.

In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from db.conexion import get_config, probar_conexion

cfg = get_config()
print('Configuración cargada:')
print(f"  Host     : {cfg['host']}:{cfg['port']}")
print(f"  Usuario  : {cfg['user']}")
print(f"  Base datos: {cfg['database']}")
print(f"  Password : {'****' if cfg['password'] else '(vacío — configura config/.env)'}")

ok, mensaje = probar_conexion()
print(mensaje)
assert ok, 'Revisa config/.env y que MySQL esté encendido'

Configuración cargada:
  Host     : localhost:3306
  Usuario  : root
  Base datos: mvnd_dw
  Password : ****
Conexión OK — MySQL 8.0.46 en localhost:3306


## Paso 2 — Leer datos limpios y armar modelo estrella

Transformación en memoria (aún **no** va a la BD).

In [3]:
from db.ingesta import preparar_datos
from pathlib import Path

RUTA_CSV = Path('../datasets/mvnd_limpio.csv')
assert RUTA_CSV.exists(), 'Ejecuta antes el notebook 02 ETL'

dims, fact, n = preparar_datos(RUTA_CSV)
print(f'Registros origen : {n:,}')
print(f'Tabla de hechos  : {len(fact):,}')
for nombre, tabla in dims.items():
    print(f'  {nombre:30} {len(tabla):>6,} filas')

Registros origen : 9,413
Tabla de hechos  : 9,413
  dim_fecha                       1,473 filas
  dim_tipo_solicitud                  3 filas
  dim_importador                    302 filas
  dim_medicamento                 1,236 filas
  dim_diagnostico                   711 filas
  dim_cantidad_categoria              4 filas


## Paso 3 — Crear esquema en MySQL (DDL)

Crea la base `mvnd_dw`, tablas `dim_*`, `fact_autorizaciones` y la vista `vw_autorizaciones_analitica`.

In [4]:
from db.ingesta import crear_esquema_bd

crear_esquema_bd()

Esquema creado/actualizado — script: 01_ddl_modelo_estrella.sql


## Paso 4 — INGESTA: cargar datos a la base de datos

**Este es el paso que inserta** `mvnd_limpio.csv` en MySQL vía `pandas.to_sql`.

In [5]:
from db.ingesta import insertar_en_mysql

insertar_en_mysql(dims, fact)

  Ingesta MySQL <- dim_fecha: 1,473 filas
  Ingesta MySQL <- dim_tipo_solicitud: 3 filas
  Ingesta MySQL <- dim_importador: 302 filas
  Ingesta MySQL <- dim_medicamento: 1,236 filas
  Ingesta MySQL <- dim_diagnostico: 711 filas
  Ingesta MySQL <- dim_cantidad_categoria: 4 filas
  Ingesta MySQL <- fact_autorizaciones: 9,413 filas

Ingesta completada en base: mvnd_dw
Vista analitica: vw_autorizaciones_analitica


## Paso 5 — Verificar filas cargadas

In [6]:
from db.ingesta import verificar_carga

verificar_carga()


--- Verificación post-ingesta ---
  dim_fecha                              1,473 filas
  dim_tipo_solicitud                         3 filas
  dim_importador                           302 filas
  dim_medicamento                        1,236 filas
  dim_diagnostico                          711 filas
  dim_cantidad_categoria                     4 filas
  fact_autorizaciones                    9,413 filas
  vw_autorizaciones_analitica            9,413 filas


## Paso 6 — Consulta de prueba (vista analítica)

Misma estructura que usará **Power BI**.

In [7]:
import pandas as pd
from db.conexion import crear_engine

engine = crear_engine()
muestra = pd.read_sql(
    """
    SELECT anio, tipo_solicitud, principio_activo, importador, capitulo_cie10, cantidad
    FROM vw_autorizaciones_analitica
    LIMIT 5
    """,
    engine,
)
muestra

,anio,tipo_solicitud,principio_activo,importador,capitulo_cie10,cantidad
0,2026,PACIENTE ESPECIFICO,ELEXACAFTOR / TEZACAFTOR / IVACAFTOR,VALENTECH PHARMA COLOMBIA SAS,"Enf. endocrinas, nutricionales y metabólicas",6.0
1,2026,MAS DE UN PACIENTE,ASPARAGINASA ERWINIA,HB HUMAN BIOSCIENCE SAS.,Neoplasias malignas (oncológico),300.0
2,2026,URGENCIA CLINICA,TEPOTINIB,GESTIFARMA SAS.,Neoplasias malignas (oncológico),3.0
3,2026,PACIENTE ESPECIFICO,AZTREONAM,GLOBAL SERVICE PHARMACEUTICAL SAS.,"Enf. endocrinas, nutricionales y metabólicas",2.0
4,2026,PACIENTE ESPECIFICO,CLORURO DE POTASIO,GLOBAL SERVICE PHARMACEUTICAL SAS.,Enfermedades del sistema genitourinario,6.0


---

### Atajo: ingesta completa en un solo comando y añadir tablas al data werehouse

```bash
python ingesta_mysql.py
```

Ejecuta: conexión → DDL → ingesta → verificación.

In [10]:

import os
import subprocess

# Ruta del proyecto
ruta_proyecto = r"C:\Users\Julian\OneDrive\Desktop\Projects\DIPLOMADO"

# Cambiar al directorio del proyecto
os.chdir(ruta_proyecto)

# Verificar ubicación actual
print("Directorio actual:")
print(os.getcwd())

# Verificar que exista el archivo
archivo = "ingesta_mysql.py"

if os.path.exists(archivo):
    print(f"\n✅ Archivo encontrado: {archivo}\n")

    # Ejecutar script
    resultado = subprocess.run(
        ["python", archivo],
        capture_output=True,
        text=True
    )

    print("=========== STDOUT ===========")
    print(resultado.stdout)

    print("\n=========== STDERR ===========")
    print(resultado.stderr)

    print("\n=========== CÓDIGO DE SALIDA ===========")
    print(resultado.returncode)

else:
    print(f"❌ No se encontró el archivo: {archivo}")

Directorio actual:
C:\Users\Julian\OneDrive\Desktop\Projects\DIPLOMADO

✅ Archivo encontrado: ingesta_mysql.py

=========== STDOUT ===========
  INGESTA MVND - CSV limpio a MySQL (modelo estrella)
ConexiÃ³n OK â€” MySQL 8.0.46 en localhost:3306
ConexiÃ³n OK â€” MySQL 8.0.46 en localhost:3306
Datos preparados desde CSV: 9,413 registros

--- Respaldo CSV (data/warehouse/) ---
  dim_fecha.csv: 1,473 filas
  dim_tipo_solicitud.csv: 3 filas
  dim_importador.csv: 302 filas
  dim_medicamento.csv: 1,236 filas
  dim_diagnostico.csv: 711 filas
  dim_cantidad_categoria.csv: 4 filas
  fact_autorizaciones.csv: 9,413 filas

Tablas exportadas en: C:\Users\Julian\OneDrive\Desktop\Projects\DIPLOMADO\data\warehouse

--- Crear esquema (DDL) ---
Esquema creado/actualizado â€” script: 01_ddl_modelo_estrella.sql

--- Ajustar columnas ---
Columnas ajustadas â€” script: 03_ajustes_columnas_datos.sql

--- Ingesta a MySQL ---
  Ingesta MySQL <- dim_fecha: 1,473 filas
  Ingesta MySQL <- dim_tipo_solicitud: 3 fil